In [ ]:
# Shared bootstrap for notebooks organized by world.
from pathlib import Path
import os, sys
_here = Path.cwd().resolve()
_roots = ( _here, *_here.parents )
_root = next((p for p in _roots if (p / 'pyproject.toml').is_file() and (p / 'src').is_dir()), None)
if _root is None: raise FileNotFoundError('Launch Jupyter from inside the research repository')
if str(_root) not in sys.path: sys.path.insert(0, str(_root))
os.chdir(_root / 'notebooks')  # preserve paths from older notebooks
print('Repository root:', _root)


# Long-budget curriculum experiment — notebook-only runner

**Question.** Do differences among `mixed`, `staged`, `staged_rehearsal`, and `reverse` persist when we give the *same model and training examples* a larger budget?

This notebook **contains the training loop, evaluation orchestration, and analysis**; the deterministic schedule planner is a reusable module in `src.medium_visual_world.curriculum`. It imports only the existing model, dataset, augmentation, and seed utilities from `src.medium_visual_world.learning`; **it does not call a separate training script**. No test manifests are read.

**Predefined protocol:** main 36 full-task-equivalent passes + common mixed 9 passes = **45 presentations per training question**; batch size 50; dataset 10,000 question rows → **9,000 updates per run**. Rehearsal uses 3 passes per earlier task at each applicable later phase (the same ratio as the 3,000-step protocol). A shared global cosine LR goes from `1e-4` to `1e-5`, with **no phase-specific resets**. Within this experiment, the same LR is applied at every global step to every schedule.

**Interpretation caveat:** relative to the prior 3,000-step constant-LR experiment, both the **budget and LR policy** change; any absolute improvement across experiments cannot be attributed to budget alone. Within *this* experiment, comparisons between curricula remain controlled.

**How to use:** start with `RUN_MODE = "plan"`. Change to `"smoke"` for a tiny run, then `"train"` with `SEEDS=[42]` before expanding to five seeds. Use a **different `OUT_DIR`** when expanding the seed set. Keep dataset hashes and protocol values fixed during the main comparison.

## 1 · Configuration
Change only this cell to configure the experiment. All steps, phases and exposures below are derived automatically.

In [ ]:
from pathlib import Path

RUN_MODE = "plan"  # "plan" | "smoke" | "train"
SEEDS = [42]  # Change to [42, 43, 44, 45, 46] after validating seed 42
SCHEDULES = ("mixed", "staged", "staged_rehearsal", "reverse")
MAIN_EPOCHS = 36
COMMON_EPOCHS = 9
REHEARSAL_EPOCHS = 3
BATCH_SIZE = 50
EVAL_EVERY = 100
INITIAL_LR = 1e-4
FINAL_LR = 1e-5
LR_POLICY = "cosine"  # "cosine" | "constant"; identical across schedules
WEIGHT_DECAY = 0.0
MODEL_DIM = 48
PATCH_SIZE = 4
AUGMENT = True
SAVE_CHECKPOINTS = False
DEVICE = "cuda"  # "cuda" | "cpu"
# Relative paths are resolved against repository root.
DATA_REL = Path("outputs/medium_world/dataset")
OUT_REL = Path("outputs/medium_world/experiments/long_budget_cosine_seed42")

assert RUN_MODE in ("plan", "smoke", "train")
assert LR_POLICY in ("cosine", "constant")
assert all(isinstance(seed, int) for seed in SEEDS) and len(SEEDS) == len(set(SEEDS))
assert MAIN_EPOCHS > 2 * REHEARSAL_EPOCHS >= 2
assert COMMON_EPOCHS >= 0 and BATCH_SIZE > 0 and EVAL_EVERY > 0
assert 0 < FINAL_LR <= INITIAL_LR

## 2 · Imports and repository discovery

In [ ]:
import sys, json, random, math, time, hashlib
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from IPython.display import display

def discover_root():
    current = Path.cwd().resolve()
    for candidate in (current, *current.parents):
        if (candidate / "src/medium_visual_world/learning.py").is_file():
            return candidate
    raise FileNotFoundError("Open this notebook from within The-order-of-Learning repository")

ROOT = discover_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.curriculum import make_blocks, TASKS as CURRICULUM_TASKS
assert TASKS == CURRICULUM_TASKS
DATA_DIR = ROOT / DATA_REL
OUT_DIR = ROOT / OUT_REL
if DEVICE == "cuda" and not torch.cuda.is_available() and RUN_MODE != "plan":
    raise RuntimeError("CUDA is unavailable; set DEVICE='cpu' for testing")
DEVICE_OBJ = torch.device("cpu" if RUN_MODE == "plan" else DEVICE)
print("Repository:", ROOT)
print("Mode:", RUN_MODE, "| device:", DEVICE_OBJ)

## 3 · Dataset integrity (no test-set access)
Validates existing manifest hashes and the dataset audit; loads training and development questions only for actual runs.

In [ ]:
def dataset_hashes(directory):
    metadata_path = directory / "dataset_metadata.json"
    audit_path = directory / "audit_report.json"
    if not metadata_path.is_file() or not audit_path.is_file():
        raise FileNotFoundError(f"Missing dataset metadata/audit: {directory}")
    metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
    audit = json.loads(audit_path.read_text(encoding="utf-8"))
    if audit.get("status") != "passed":
        raise RuntimeError("Dataset audit has not passed")
    hashes = {}
    for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl"):
        path = directory / name
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        if digest != metadata["manifests_sha256"][name]:
            raise ValueError(f"Manifest hash mismatch: {name}")
        hashes[name] = digest
    return hashes

if RUN_MODE == "plan" and not DATA_DIR.exists():
    print("Data directory not mounted here; planning will use synthetic task labels.")
    HASHES = None
else:
    HASHES = dataset_hashes(DATA_DIR)
    print("Verified:", {name: value[:12] for name, value in HASHES.items()})

## 4 · Reproducible schedules and **exact per-example exposure**
All four methods contain 45 presentations **per individual training question**, not merely comparable task percentages. The same base `seed` initializes the same model across schedules.

In [ ]:
def build_schedule(rows, schedule, seed):
    blocks, plan = make_blocks(rows, schedule, MAIN_EPOCHS, COMMON_EPOCHS,
                               seed, REHEARSAL_EPOCHS)
    if any(len(ids) % BATCH_SIZE for _, ids in blocks):
        raise ValueError("Partial phase batch; change epochs or batch size")
    details = []
    for phase, ids in blocks:
        counts = Counter(rows[i]["template"] for i in ids)
        details.append({"phase": phase, "updates": len(ids) // BATCH_SIZE,
                        "presentations": len(ids),
                        **{task: counts[task] for task in TASKS}})
    return blocks, details

# The planning preview also runs without local dataset files.
if DATA_DIR.joinpath("train.jsonl").is_file():
    from src.medium_visual_world.generate import read_jsonl
    planning_rows = list(read_jsonl(DATA_DIR / "train.jsonl"))
else:
    planning_rows = [{"template": task} for task in TASKS for _ in range(2500)]
plans = {}
for schedule in SCHEDULES:
    blocks, details = build_schedule(planning_rows, schedule, SEEDS[0])
    plans[schedule] = details
    assert sum(len(ids) for _, ids in blocks) == len(planning_rows) * (MAIN_EPOCHS + COMMON_EPOCHS)
    assert sum(row["updates"] for row in details) == len(planning_rows) * (MAIN_EPOCHS + COMMON_EPOCHS) // BATCH_SIZE
print("Expected training updates:", len(planning_rows) * (MAIN_EPOCHS + COMMON_EPOCHS) // BATCH_SIZE)
display(pd.concat([pd.DataFrame(details).assign(schedule=schedule)
                   for schedule, details in plans.items()], ignore_index=True))

## 5 · Shared learning-rate schedule
The LR depends **only on global optimizer update count**, not the phase or evaluation loss. All curricula receive an identical trajectory. AdamW state is **not** reset at boundaries.

In [ ]:
def learning_rate(update_index, total_updates):
    if LR_POLICY == "constant":
        return INITIAL_LR
    progress = (update_index - 1) / max(total_updates - 1, 1)
    return FINAL_LR + 0.5 * (INITIAL_LR - FINAL_LR) * (1 + math.cos(math.pi * progress))

total_updates = len(planning_rows) * (MAIN_EPOCHS + COMMON_EPOCHS) // BATCH_SIZE
preview_steps = np.linspace(1, total_updates, 250, dtype=int)
plt.figure(figsize=(8, 3))
plt.plot(preview_steps, [learning_rate(int(k), total_updates) for k in preview_steps])
plt.xlabel("Global optimizer update")
plt.ylabel("Learning rate")
plt.title(f"Shared {LR_POLICY} LR policy — same for all schedules")
plt.grid(alpha=0.2)
plt.show()
print(f"Update 1: {learning_rate(1, total_updates):.2g}; final: {learning_rate(total_updates, total_updates):.2g}")

## 6 · Evaluation and training implemented **inside this notebook**
ID-dev is used to record loss and task accuracy. OOD-dev is diagnostic only. **ID-test and OOD-test are not touched.** Accuracy at fixed final update is primary; the best ID-dev-loss checkpoint is additional, not substituted for the final model.

In [ ]:
@torch.inference_mode()
def evaluate_here(model, dataset, device, batch_size, limit=None, constant_image=False):
    model.eval()
    n = min(len(dataset), limit) if limit is not None else len(dataset)
    hits, total_loss = 0, 0.0
    counts, correct_by_task = Counter(), Counter()
    for start in range(0, n, batch_size):
        ids = list(range(start, min(start + batch_size, n)))
        x, q, y = dataset.batch(ids, device, constant_image)
        logits = model(x, q)
        total_loss += nn.functional.cross_entropy(logits, y, reduction="sum").item()
        correct = (logits.argmax(1) == y).tolist()
        for i, is_correct in zip(ids, correct):
            task = dataset.tasks[i]
            counts[task] += 1
            correct_by_task[task] += int(is_correct)
            hits += int(is_correct)
    return {"n": n, "accuracy": hits / n, "loss": total_loss / n,
            "by_template": {task: correct_by_task[task] / counts[task]
                             for task in counts if counts[task]}}

def curve_area(history, task="pair"):
    points = sorted({entry["step"]: entry["id_dev"]["by_template"][task]
                     for entry in history}.items())
    if len(points) < 2 or points[0][0] != 0:
        raise ValueError("Need evaluation at step 0")
    width = points[-1][0]
    return sum((b-a) * (x+y) / 2
               for (a, x), (b, y) in zip(points[:-1], points[1:])) / width

def run_one(schedule, seed, train, dev, ood, out, mode):
    set_seed(seed)
    smoke = mode == "smoke"
    dim, patch = (16, 8) if smoke else (MODEL_DIM, PATCH_SIZE)
    model = AttributeBindingViT(dim=dim, patch_size=patch).to(DEVICE_OBJ)
    optimizer = torch.optim.AdamW(model.parameters(), lr=INITIAL_LR,
                                  weight_decay=WEIGHT_DECAY)
    blocks, phase_details = build_schedule(train.rows, schedule, seed)
    intended_updates = sum(len(ids) // BATCH_SIZE for _, ids in blocks)
    checkpoint_dir = out / "checkpoints"
    history, loss_sum, loss_count, step = [], 0.0, 0, 0
    best_loss, best_step = float("inf"), -1
    eval_limit = min(64, len(dev), len(ood)) if smoke else None
    def record(phase):
        nonlocal loss_sum, loss_count, best_loss, best_step
        id_result = evaluate_here(model, dev, DEVICE_OBJ, BATCH_SIZE, eval_limit)
        ood_result = evaluate_here(model, ood, DEVICE_OBJ, BATCH_SIZE, eval_limit)
        history.append({"step": step, "phase": phase,
                        "batch_train_loss": loss_sum / loss_count if loss_count else None,
                        "lr": optimizer.param_groups[0]["lr"],
                        "id_dev": id_result, "ood_dev": ood_result})
        if id_result["loss"] < best_loss:
            best_loss, best_step = id_result["loss"], step
            if SAVE_CHECKPOINTS and not smoke:
                checkpoint_dir.mkdir(parents=True, exist_ok=True)
                torch.save({"model": model.state_dict(), "step": step,
                            "seed": seed, "schedule": schedule, "dim": dim,
                            "patch_size": patch}, checkpoint_dir / f"{schedule}_{seed}_best.pt")
        loss_sum, loss_count = 0.0, 0
        pair = id_result["by_template"].get("pair", float("nan"))
        print(f"{schedule:17} seed={seed} step={step:5} phase={phase:12} "
              f"ID-pair={pair:.3f} overall={id_result['accuracy']:.3f} "
              f"LR={optimizer.param_groups[0]['lr']:.2g}", flush=True)
    record("initial")
    for phase, ids in blocks:
        phase_updates = 0
        for start in range(0, len(ids), BATCH_SIZE):
            if smoke and phase_updates >= 2:
                break
            batch_ids = ids[start:start+BATCH_SIZE]
            x, q, y = train.batch(batch_ids, DEVICE_OBJ)
            model.train()
            if AUGMENT:
                x = augment_images(x)
            lr = learning_rate(step + 1, intended_updates)
            for group in optimizer.param_groups:
                group["lr"] = lr
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x, q), y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            phase_updates += 1
            loss_sum += loss.item()
            loss_count += 1
            if not smoke and step % EVAL_EVERY == 0:
                record(phase)
        if history[-1]["step"] != step:
            record(phase)
    final = {"step": step, "phase": "final",
             "id_dev": evaluate_here(model, dev, DEVICE_OBJ, BATCH_SIZE, eval_limit),
             "ood_dev": evaluate_here(model, ood, DEVICE_OBJ, BATCH_SIZE, eval_limit)}
    final["id_dev_constant_image"] = evaluate_here(model, dev, DEVICE_OBJ,
                                                     BATCH_SIZE, eval_limit, True)
    if SAVE_CHECKPOINTS and not smoke:
        checkpoint_dir.mkdir(parents=True, exist_ok=True)
        torch.save({"model": model.state_dict(), "step": step, "seed": seed,
                    "schedule": schedule, "dim": dim, "patch_size": patch},
                   checkpoint_dir / f"{schedule}_{seed}_final.pt")
    return {"seed": seed, "schedule": schedule,
            "status": "smoke_only" if smoke else "completed",
            "steps_run": step, "planned_updates": intended_updates,
            "phase_plan": phase_details,
            "total_presentations": sum(len(ids) for _, ids in blocks),
            "presentations_per_example": MAIN_EPOCHS + COMMON_EPOCHS,
            "history": history, "final": final,
            "best_id_dev_loss": best_loss, "best_id_dev_step": best_step,
            "id_pair_aulc": curve_area(history) if step else None}

## 7 · Run and save incrementally
The output is written after each completed run, so completed seeds survive interruptions. The notebook refuses to combine results with an existing file whose protocol differs. In `plan` mode this cell only displays instructions.

In [ ]:
def protocol():
    return {"seeds": list(SEEDS), "schedules": list(SCHEDULES),
            "main_epochs": MAIN_EPOCHS, "common_epochs": COMMON_EPOCHS,
            "rehearsal_epochs": REHEARSAL_EPOCHS, "batch_size": BATCH_SIZE,
            "eval_every": EVAL_EVERY, "lr_policy": LR_POLICY,
            "initial_lr": INITIAL_LR, "final_lr": FINAL_LR,
            "weight_decay": WEIGHT_DECAY, "dim": MODEL_DIM,
            "patch_size": PATCH_SIZE, "augment": AUGMENT,
            "total_updates": total_updates}

def write_report(path, report):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(report, indent=2) + "\n", encoding="utf-8")
    tmp.replace(path)

if RUN_MODE == "plan":
    print("Plan verified. Set RUN_MODE='smoke' to test or 'train' to train.")
else:
    torch.set_num_threads(min(4, torch.get_num_threads()))
    out = OUT_DIR if RUN_MODE == "train" else ROOT / "outputs/medium_world/experiments/long_budget_smoke"
    report_path = out / ("schedule_comparison.json" if RUN_MODE == "train" else "smoke.json")
    if report_path.exists():
        report = json.loads(report_path.read_text(encoding="utf-8"))
        if report["config"] != protocol() or report["dataset_hashes"] != HASHES or report["run_mode"] != RUN_MODE:
            raise RuntimeError("Existing output uses different config/data; choose a fresh OUT_REL")
    else:
        report = {"status": "exploratory" if RUN_MODE == "train" else "smoke_only",
                  "run_mode": RUN_MODE, "dataset_hashes": HASHES,
                  "environment": environment(), "config": protocol(), "runs": []}
    train = MediumDataset(DATA_DIR / "train.jsonl")
    dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
    ood = MediumDataset(DATA_DIR / "ood_dev.jsonl")
    if len(train) * (MAIN_EPOCHS + COMMON_EPOCHS) != total_updates * BATCH_SIZE:
        raise AssertionError("Actual question count differs from planning count")
    completed = {(r["seed"], r["schedule"]) for r in report["runs"]}
    for seed in SEEDS:
        for schedule in SCHEDULES:
            if (seed, schedule) in completed:
                print("Already complete:", seed, schedule)
                continue
            result = run_one(schedule, seed, train, dev, ood, out, RUN_MODE)
            report["runs"].append(result)
            write_report(report_path, report)
    print("Saved results:", report_path)

## 8 · Results analysis (if completed results exist)
The plots and tables are generated **in the same notebook**. They include paired seed comparisons and learning curves. These are development-set results; no final test evaluation is performed.

In [ ]:
result_path = OUT_DIR / "schedule_comparison.json"
if not result_path.is_file():
    print("No full-run result found yet:", result_path)
    report_loaded, metrics = None, pd.DataFrame()
else:
    report_loaded = json.loads(result_path.read_text(encoding="utf-8"))
    rows = []
    for run in report_loaded["runs"]:
        if run["status"] != "completed":
            continue
        f = run["final"]
        rows.append({"seed": run["seed"], "schedule": run["schedule"],
                     "overall_id": f["id_dev"]["accuracy"],
                     "pair_id": f["id_dev"]["by_template"].get("pair", np.nan),
                     "pair_ood": f["ood_dev"]["by_template"].get("pair", np.nan),
                     "pair_aulc": run["id_pair_aulc"],
                     **{f"id_{task}": f["id_dev"]["by_template"].get(task, np.nan)
                        for task in TASKS}})
    metrics = pd.DataFrame(rows)
    display(metrics.sort_values(["seed", "schedule"]).reset_index(drop=True))
    summary = metrics.groupby("schedule")[["overall_id", "pair_id", "pair_ood", "pair_aulc"]].agg(["mean", "std"])
    display(summary)
    for metric in ("pair_id", "pair_ood", "pair_aulc", "overall_id"):
        pivot = metrics.pivot(index="seed", columns="schedule", values=metric)
        if {"mixed", "staged_rehearsal"}.issubset(pivot.columns):
            differences = (pivot["staged_rehearsal"] - pivot["mixed"]).dropna()
            print(f"Rehearsal - mixed, {metric}: mean={differences.mean():+.4f}, "
                  f"wins={(differences > 0).sum()}/{len(differences)}")

In [ ]:
if report_loaded is not None:
    import matplotlib.pyplot as plt
    for task, title in (("pair", "ID binding accuracy"), ("overall", "Overall ID accuracy")):
        plt.figure(figsize=(9, 4))
        for schedule in SCHEDULES:
            curves = []
            for run in report_loaded["runs"]:
                if run["schedule"] != schedule or run["status"] != "completed":
                    continue
                points = {r["step"]: r["id_dev"]["accuracy"] if task == "overall"
                          else r["id_dev"]["by_template"].get("pair", np.nan)
                          for r in run["history"]}
                curves.append(points)
            if not curves:
                continue
            aligned_steps = sorted(set.intersection(*(set(c) for c in curves)))
            values = np.array([[c[s] for s in aligned_steps] for c in curves])
            mean = values.mean(axis=0)
            std = values.std(axis=0, ddof=1) if len(values) > 1 else np.zeros(len(mean))
            plt.plot(aligned_steps, mean, label=schedule)
            plt.fill_between(aligned_steps, mean-std, mean+std, alpha=0.12)
        plt.axhline(0.5, linestyle="--", linewidth=1, alpha=0.4)
        plt.title(title + " — mean ± seed SD")
        plt.xlabel("Global optimizer updates")
        plt.ylabel("Accuracy")
        plt.legend()
        plt.grid(alpha=0.15)
        plt.show()

## 9 · Sanity checks and interpretation checklist

Before drawing a conclusion:

- Check every schedule has **9,000 steps** and **45 presentations per individual question**.
- Confirm `cosine` LR is identical by global step (no restarts) and AdamW is never reset.
- Compare matched-seed **final ID binding**, **binding AULC**, and **OOD binding**. Report seed variation; five seeds is still a small sample.
- Compare retention *at matched cumulative exposure*, ideally step 7,200 (end of main phases, before common consolidation), not after different numbers of updates into different phases.
- Check shape accuracy alongside binding, since shape learning is a limiting factor.
- The longer-run results are a new experiment: changes vs 3,000-step runs may reflect **both extra training and LR decay**.
- Keep `id_test.jsonl` and `ood_test.jsonl` untouched until a final, frozen evaluation protocol is chosen.

**Next independent experiment:** mastery-based progression with a maximum budget, stopping criteria on ID-dev, and an exposure-matched mixed control. Avoid implementing mastery and LR restarts at the same time.